# SageAgent v5 Production UI

SageAgent v5 is the production-ready SageMaker notebook coding agent. It keeps the v4-style notebook UI, but runs the rebuilt v5 engine underneath: Bedrock Claude models, durable memory/status, subagents, checkpoints, compaction, result replay, telemetry, and verify/done gates.

**Run cells 1-3 in order:**
- **Cell 1** installs packages, usually once per kernel.
- **Cell 2** sets model, AWS region, mock mode, thinking mode, and budget settings.
- **Cell 3** launches the chat UI.

**Core runtime files in the production zip:** `chat.ipynb`, `entry.py`, `sagemaker_agent.py`, `agent.py`, `commands.py`, `memory.md`, `AGENT_STATUS.md`, and the `core/`, `runtime/`, `tools/`, `prompt/`, `skills/`, `subagent/`, `security/`, and `ui/` packages.

**Companion guide:** open `chat.md` for the user guide, command list, skills list, safety notes, and troubleshooting.

In [ ]:
# Cell 1: install dependencies (run once per kernel)
!pip install -q boto3 ipywidgets Pillow

In [ ]:
# Locate the v5 runtime even if Jupyter was launched from the repo root.
import sys
from pathlib import Path

def _ensure_sageagent_path():
    start = Path.cwd().resolve()
    candidates = []
    for base in (start, *start.parents):
        candidates.extend([
            base,
            base / "compact_v5",
            base / "MAIN" / "agent",
            base / "compact_v5" / "MAIN" / "agent",
            base / "sagemaker-coding-agent" / "compact_v5",
            base / "sagemaker-coding-agent" / "compact_v5" / "MAIN" / "agent",
        ])
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "entry.py").is_file():
            path = str(candidate)
            if path not in sys.path:
                sys.path.insert(0, path)
            return candidate
    raise ModuleNotFoundError(
        "Could not locate entry.py. Open chat.ipynb from the shipped zip root "
        "or compact_v5/MAIN/agent, or set PYTHONPATH to the folder containing entry.py."
    )

_AGENT_ROOT = _ensure_sageagent_path()
print(f"Using SageAgent runtime: {_AGENT_ROOT}")

# Cell 2: configure
from entry import CONFIG, BEDROCK_MODELS

# Model: pick from BEDROCK_MODELS or set your own.
CONFIG.model_id = BEDROCK_MODELS[1][1]  # default Haiku 4.5
CONFIG.region = "us-east-1"

# Mock mode: True = no real Bedrock calls (use for first run / smoke test).
CONFIG.mock_mode = True

# Thinking mode: True = send thinking config every call.
# UI surfaces toggle + budget. Default OFF.
CONFIG.thinking_enabled = False
CONFIG.thinking_budget = 4096

# Skill auto-trigger default-OFF: False = skills only via /skill use <name>.
CONFIG.enable_skill_auto_trigger = False

print(f'Configured: model={CONFIG.model_id}, mock_mode={CONFIG.mock_mode}, thinking={CONFIG.thinking_enabled}')


In [ ]:
# Locate the v5 runtime even if Jupyter was launched from the repo root.
import sys
from pathlib import Path

def _ensure_sageagent_path():
    start = Path.cwd().resolve()
    candidates = []
    for base in (start, *start.parents):
        candidates.extend([
            base,
            base / "compact_v5",
            base / "MAIN" / "agent",
            base / "compact_v5" / "MAIN" / "agent",
            base / "sagemaker-coding-agent" / "compact_v5",
            base / "sagemaker-coding-agent" / "compact_v5" / "MAIN" / "agent",
        ])
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "entry.py").is_file():
            path = str(candidate)
            if path not in sys.path:
                sys.path.insert(0, path)
            return candidate
    raise ModuleNotFoundError(
        "Could not locate entry.py. Open chat.ipynb from the shipped zip root "
        "or compact_v5/MAIN/agent, or set PYTHONPATH to the folder containing entry.py."
    )

_AGENT_ROOT = _ensure_sageagent_path()
print(f"Using SageAgent runtime: {_AGENT_ROOT}")

# Cell 3: launch chat UI
from entry import create_chat_ui
from IPython.display import display

ui = create_chat_ui()
display(ui.render())


---

## Quick Reference

**Buttons:** Send, Stop, Clear. If widgets are unavailable, use the console fallback: `ui.send("your message")`.

**Long-running work flow:** keep `AGENT_STATUS.md` and `memory.md` fresh, use `/save`, `/resume`, and `/checkpoint` for handoff/rollback, then use `/verify` and `/done` before trusting a completed software task.

**Core commands:** `/status`, `/save`, `/resume`, `/checkpoint`, `/cost`, `/context`, `/verify`, `/done`, `/dream`, `/skills`, `/skill use`, `/skill clear`, `/skill apply`, `/skill reject`, `/skillify`, `/promote-to-skill`, `/phase`, `/diffs`, `/regression`, `/revert`, `/auth`, `/quit`.

**Production skills:** batch, clara, debug, design, html, init, init-verifiers, reflexion, remember, report, review, security-review, simplify, skillify, verify. Use `/skills` to list them and `/skill use <name>` to activate one.

**Cost and context:** `/cost` shows token/cache/model spend. `/context` shows context pressure. v5 tracks parent/subagent usage and blocks unsafe done claims through the verification gates.

**Memory:** `memory.md` is auto-loaded as durable memory. `/dream` consolidates memory when you want a cleanup pass.

**Ship evidence:** v5.0.1 passed the final R-tier gate and final Claude production-readiness review. Test evidence is kept outside the production zip under `_status/`.